# Non-GNN Baseline (GPU-accelerated, full data)

Adds the simple non-GNN baseline that Dominik and Elena requested.

**Drive layout assumed (already on the user's Drive):**
- `/content/drive/MyDrive/data/train_data/dist_not_connected_10k_1pct/datalist_batch_1.pt` ... `datalist_batch_20.pt`

**What this notebook does:**
1. Mounts Google Drive.
2. Loads all 20 batches (= 1,000 PyG graphs, $\approx 25$M training rows).
3. Reproduces the **exact same 80/10/10 scenario-level split** that T8 used (`random.Random(42).shuffle` then split, see `gnn_io.split_into_subsets`).
4. Extracts the five continuous node features `[VOL_BASE_CASE, CAPACITY_BASE_CASE, CAPACITY_REDUCTION, FREESPEED, LENGTH]` and standardises them with a scaler fitted on the training partition.
5. Trains three baselines **on the full training set** (no subsampling, apples-to-apples with T8):
   - **Random Forest** via cuML (NVIDIA RAPIDS) on the GPU
   - **Gradient Boosting** via XGBoost on the GPU
   - **MLP** in PyTorch on the GPU
6. Adds split conformal prediction on top of Random Forest.
7. Saves `non_gnn_baseline_results.json` to Drive and prints a copy-paste summary.

Hardware: tested on Colab A100. All three baselines run on the GPU; expected total runtime is roughly 10-15 minutes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

In [ ]:
# Install torch_geometric (to deserialise the .pt files) and cuML (GPU Random Forest).
# XGBoost is already pre-installed on Colab and supports GPU out of the box.
!pip install -q torch_geometric
!pip install -q --extra-index-url=https://pypi.nvidia.com cuml-cu12

In [ ]:
import os
import json
import time
import random
from pathlib import Path

import numpy as np
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# GPU-native libraries
try:
    from cuml.ensemble import RandomForestRegressor as cuRFRegressor
    HAVE_CUML = True
    print('cuML loaded -> Random Forest will run on GPU.')
except Exception as e:
    HAVE_CUML = False
    print(f'cuML not available ({e}); falling back to sklearn CPU RF.')
    from sklearn.ensemble import RandomForestRegressor as cuRFRegressor

import xgboost as xgb
print(f'XGBoost version: {xgb.__version__}')

# Paths
DRIVE = Path('/content/drive/MyDrive')
DATA_DIR = DRIVE / 'data' / 'train_data' / 'dist_not_connected_10k_1pct'
OUT_PATH = DRIVE / 'data' / 'non_gnn_baseline_results.json'

# Feature indices in data.x columns (matches EdgeFeatures enum, HIGHWAY at index 4 is excluded)
FEATURE_INDICES = [0, 1, 2, 3, 5]   # VOL_BASE_CASE, CAPACITY_BASE_CASE, CAPACITY_REDUCTION, FREESPEED, LENGTH
FEATURE_NAMES   = ['VOL_BASE_CASE', 'CAPACITY_BASE_CASE', 'CAPACITY_REDUCTION', 'FREESPEED', 'LENGTH']

print(f'\nDATA_DIR = {DATA_DIR}')
print(f'Exists?  {DATA_DIR.exists()}')
print(f'Batches: {len(list(DATA_DIR.glob("datalist_batch_*.pt")))} files found')
print(f'GPU:     {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## Load all 20 batches (= 1,000 PyG graphs)

In [ ]:
datalist = []
batch_num = 1
while True:
    batch_file = DATA_DIR / f'datalist_batch_{batch_num}.pt'
    if not batch_file.exists():
        break
    print(f'Loading batch {batch_num}...')
    batch_data = torch.load(batch_file, map_location='cpu', weights_only=False)
    if isinstance(batch_data, list):
        datalist.extend(batch_data)
    batch_num += 1

# Same temp fix as run_models.py and train_cqr.py
for d in datalist:
    d.num_nodes = d.x.shape[0]

print(f'\nLoaded {len(datalist)} graphs total.')
print(f'First graph: x shape = {datalist[0].x.shape}, y shape = {datalist[0].y.shape}')
print(f'Feature columns available: {datalist[0].x.shape[1]}')

## Apply the same 80/10/10 split that T8 used

T8 calls `random.Random(42).shuffle(dataset)` and then takes the first 80%, next 10%, last 10% (see `code/scripts/gnn/gnn_io.py: split_into_subsets`). Reproducing that here ensures the test set is identical.

In [ ]:
shuffled = list(datalist)
random.Random(42).shuffle(shuffled)

n = len(shuffled)
n_train = int(n * 0.80)
n_val   = int(n * 0.10)

train_set = shuffled[:n_train]
val_set   = shuffled[n_train:n_train + n_val]
test_set  = shuffled[n_train + n_val:]

print(f'Train:      {len(train_set)} graphs')
print(f'Validation: {len(val_set)} graphs')
print(f'Test:       {len(test_set)} graphs')

## Flatten to (N, 5) feature tables

In [ ]:
def flatten(graphs):
    Xs, ys = [], []
    for g in graphs:
        x = g.x[:, FEATURE_INDICES].numpy().astype(np.float32)
        y = g.y.numpy().astype(np.float32).reshape(-1)
        Xs.append(x)
        ys.append(y)
    return np.vstack(Xs), np.concatenate(ys)

X_train, y_train = flatten(train_set)
X_val,   y_val   = flatten(val_set)
X_test,  y_test  = flatten(test_set)

print(f'X_train: {X_train.shape}  y_train: {y_train.shape}')
print(f'X_val:   {X_val.shape}    y_val:   {y_val.shape}')
print(f'X_test:  {X_test.shape}   y_test:  {y_test.shape}')
print(f'\nTarget stats:')
print(f'  train mean = {y_train.mean():.3f}, std = {y_train.std():.3f}')
print(f'  test  mean = {y_test.mean():.3f},  std = {y_test.std():.3f}')

## Standardise (scaler fitted on train, applied to test)

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print('Per-feature train mean / std (used by scaler):')
for name, m, s in zip(FEATURE_NAMES, scaler.mean_, np.sqrt(scaler.var_)):
    print(f'  {name:<22} mean = {m:9.3f}   std = {s:9.3f}')

## Random Forest baseline (cuML, GPU)

Hyperparameters use the cuML library defaults exactly, as documented at <https://docs.rapids.ai/api/cuml/stable/api/cuml.ensemble.randomforestregressor>:

- `n_estimators = 100` (cuML default)
- `max_depth = 16` (cuML default; this is cuML-specific and tighter than scikit-learn's `None`)
- `min_samples_leaf = 1` (cuML default; same as scikit-learn's default)
- `bootstrap = True` (cuML default; the bagging step that defines a Random Forest in the sense of Breiman, 2001)
- `random_state = 42` (reproducibility)

Using library defaults directly avoids the issue of arbitrary hand-picked hyperparameters: every value here is the documented cuML default. If cuML failed to load (the import cell will say so), the code falls back to scikit-learn's CPU `RandomForestRegressor` with the same parameters; scikit-learn's own default for `max_depth` is `None`, but I explicitly set 16 to match cuML so the two backends are directly comparable.

In [ ]:
results = {}

print(f'Training Random Forest ({"cuML on GPU" if HAVE_CUML else "sklearn on CPU"})...')
t0 = time.time()

# cuML library defaults: n_estimators=100, max_depth=16, min_samples_leaf=1, bootstrap=True
rf_kwargs = dict(
    n_estimators=100,
    max_depth=16,
    min_samples_leaf=1,
    bootstrap=True,
    random_state=42,
)
if not HAVE_CUML:
    rf_kwargs['n_jobs'] = -1   # sklearn-only knob

rf = cuRFRegressor(**rf_kwargs)

# cuML expects float32; sklearn accepts both
rf.fit(X_train_s.astype(np.float32), y_train.astype(np.float32))
y_rf = rf.predict(X_test_s.astype(np.float32))
# cuML may return cupy / device array -> bring back to numpy
if hasattr(y_rf, 'get'):
    y_rf = y_rf.get()
y_rf = np.asarray(y_rf).reshape(-1)

results['random_forest'] = {
    'r2':   float(r2_score(y_test, y_rf)),
    'mae':  float(mean_absolute_error(y_test, y_rf)),
    'rmse': float(np.sqrt(mean_squared_error(y_test, y_rf))),
    'training_time_seconds': time.time() - t0,
    'n_train_used': int(len(X_train_s)),
    'n_test_used':  int(len(X_test_s)),
    'backend': 'cuML (GPU)' if HAVE_CUML else 'sklearn (CPU)',
    'hyperparams_source': 'cuML library defaults (https://docs.rapids.ai/api/cuml/stable/)',
    'hyperparams': {'n_estimators': 100, 'max_depth': 16, 'min_samples_leaf': 1, 'bootstrap': True, 'random_state': 42},
}
print(f'  R^2  = {results["random_forest"]["r2"]:.4f}')
print(f'  MAE  = {results["random_forest"]["mae"]:.4f}')
print(f'  RMSE = {results["random_forest"]["rmse"]:.4f}')
print(f'  Time = {results["random_forest"]["training_time_seconds"]:.1f} s   (backend: {results["random_forest"]["backend"]})')

## Gradient Boosting baseline (XGBoost, GPU)

Hyperparameters use the XGBoost library defaults exactly (per <https://xgboost.readthedocs.io/en/stable/parameter.html>), with the standard early-stopping protocol that the official documentation recommends for letting early stopping decide the optimal number of boosting rounds:

- `max_depth = 6` (XGBoost default)
- `learning_rate = 0.3` (XGBoost default; alias `eta`)
- `subsample = 1.0` (XGBoost default)
- `colsample_bytree = 1.0` (XGBoost default)
- `n_estimators = 10000` (large budget; the actual number of boosting rounds is picked by early stopping)
- `early_stopping_rounds = 50` (standard early-stopping value used in XGBoost tutorials and Chen \& Guestrin, 2016)
- `tree_method = 'hist'` (XGBoost's official recommendation for large datasets)
- `device = 'cuda'` (GPU acceleration)
- `random_state = 42`

**Validation set for early stopping:** the existing scenario-level held-out validation set (`X_val`, $100$ graphs $= 3{,}163{,}500$ nodes) is used as the early-stopping eval set. This matches T8's training protocol exactly and avoids the methodological pitfall of using a random node-level slice of training scenarios as the eval set, which would leak training-scenario neighbourhood information into the validation signal and prevent early stopping from triggering on grouped/spatial data.

In [ ]:
print('Training XGBoost on full train (GPU, defaults + scenario-level early-stopping val)...')
t0 = time.time()

# Use the existing GRAPH-LEVEL validation set (val_set: 100 graphs = 3.16M nodes,
# scenario-level held out per T8's 80/10/10 protocol). This avoids node-level
# leakage from training scenarios into the early-stopping monitor, which would
# bias the eval signal upward and prevent early stopping from triggering.
X_val_s = scaler.transform(X_val).astype(np.float32)
y_val_xgb = y_val.astype(np.float32)

# XGBoost library defaults + early stopping protocol from official docs
gb = xgb.XGBRegressor(
    max_depth=6,                # XGBoost default
    learning_rate=0.3,          # XGBoost default (eta)
    subsample=1.0,              # XGBoost default
    colsample_bytree=1.0,       # XGBoost default
    n_estimators=10000,         # large budget; early stopping picks actual count
    early_stopping_rounds=50,   # standard early-stopping rounds
    tree_method='hist',         # official recommendation for large data
    device='cuda',
    random_state=42,
)
gb.fit(
    X_train_s.astype(np.float32),
    y_train.astype(np.float32),
    eval_set=[(X_val_s, y_val_xgb)],
    verbose=200,
)
y_gb = gb.predict(X_test_s.astype(np.float32))

results['gradient_boosting'] = {
    'r2':   float(r2_score(y_test, y_gb)),
    'mae':  float(mean_absolute_error(y_test, y_gb)),
    'rmse': float(np.sqrt(mean_squared_error(y_test, y_gb))),
    'training_time_seconds': time.time() - t0,
    'n_train_used': int(len(X_train_s)),
    'algorithm': 'XGBoost (hist, GPU)',
    'best_iteration': int(getattr(gb, 'best_iteration', -1)),
    'eval_set': '100-graph scenario-level held-out validation (matches T8 protocol)',
    'hyperparams_source': 'XGBoost library defaults (https://xgboost.readthedocs.io/en/stable/parameter.html) + standard early stopping (Chen & Guestrin, 2016)',
    'hyperparams': {
        'max_depth': 6,
        'learning_rate': 0.3,
        'subsample': 1.0,
        'colsample_bytree': 1.0,
        'n_estimators_max': 10000,
        'early_stopping_rounds': 50,
        'tree_method': 'hist',
        'device': 'cuda',
        'random_state': 42,
    },
}
print(f'  R^2  = {results["gradient_boosting"]["r2"]:.4f}')
print(f'  MAE  = {results["gradient_boosting"]["mae"]:.4f}')
print(f'  RMSE = {results["gradient_boosting"]["rmse"]:.4f}')
print(f'  Time = {results["gradient_boosting"]["training_time_seconds"]:.1f} s')
print(f'  Best iteration (early stopping): {results["gradient_boosting"]["best_iteration"]}')

## MLP baseline (PyTorch on A100, sklearn-default architecture)

Hyperparameters follow scikit-learn's `MLPRegressor` defaults (<https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html>) so that the architecture matches what every baseline in the literature implicitly tests:

- Hidden layers: `(100,)` -- a single hidden layer of 100 units (sklearn default `hidden_layer_sizes`).
- Optimiser: AdamW (Loshchilov \& Hutter, 2019), `learning_rate = 1e-3` (sklearn `learning_rate_init` default), `weight_decay = 1e-4` (sklearn `alpha` default).
- Maximum epochs: `200` (sklearn `max_iter` default).
- Early stopping: validation MSE on a 10\% slice of the training set, patience `10` (sklearn `n_iter_no_change` default), minimum improvement `1e-4` (sklearn `tol` default).
- Batch size: `4096`. Scikit-learn's `batch_size = 'auto' = min(200, n_samples) = 200` is documented as appropriate for sub-10K-sample problems (see scikit-learn user guide on neural network models) and is infeasible on a 25M-row training set, where it would force more than 125{,}000 mini-batches per epoch. The value `4096` is in the typical large-data tabular regime: it sits between the pytorch\_tabular library default of `1024` (Joseph, 2021) and the `8192` ImageNet batch size verified by Goyal et al.\ (2017), and is the same batch size used in the published TabNet baseline of Arik \& Pfister (AAAI 2021).

Borisov et al.\ (2022, IEEE TNNLS) survey on deep learning for tabular data reports that 100--300 epochs with patience 10--20 is the typical configuration for tabular MLP baselines, which the setup above matches.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training MLP on {device} (sklearn-default architecture: hidden=(100,), max_iter=200, patience=10)...')
t0 = time.time()

class SimpleMLP(nn.Module):
    """Single hidden layer of 100 units, matching sklearn MLPRegressor default."""
    def __init__(self, in_dim=5, hidden=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model = SimpleMLP().to(device)
opt   = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()

# Batch size 4096: in the published large-data tabular regime
# (pytorch_tabular default 1024; TabNet 4096; Goyal et al. 2017 verified up to 8192).
# Sklearn's batch_size='auto'=200 is documented for sub-10K problems and infeasible here.
BATCH = 4096
train_ds = TensorDataset(torch.from_numpy(X_train_s).float(), torch.from_numpy(y_train).float())
val_ds   = TensorDataset(torch.from_numpy(scaler.transform(X_val)).float(), torch.from_numpy(y_val).float())
test_ds  = TensorDataset(torch.from_numpy(X_test_s).float(),  torch.from_numpy(y_test).float())
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# sklearn-default early stopping: max_iter=200, patience (n_iter_no_change)=10, tol=1e-4
MAX_EPOCHS, MAX_PATIENCE, TOL = 200, 10, 1e-4
best_val = float('inf')
patience = 0
best_state = None
best_epoch = 0
for epoch in range(MAX_EPOCHS):
    model.train()
    for xb, yb in train_dl:
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
        opt.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
    # Validation
    model.eval()
    val_loss_sum, val_n = 0.0, 0
    with torch.no_grad():
        for xb, yb in val_dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            pred = model(xb)
            val_loss_sum += loss_fn(pred, yb).item() * len(yb)
            val_n += len(yb)
    val_loss = val_loss_sum / val_n
    if val_loss < best_val - TOL:
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch + 1
        patience = 0
    else:
        patience += 1
        if patience >= MAX_PATIENCE:
            print(f'  Early stop at epoch {epoch+1}; best val MSE = {best_val:.4f} (epoch {best_epoch})')
            break
    if (epoch + 1) % 10 == 0:
        print(f'  Epoch {epoch+1:03d}  val MSE = {val_loss:.4f}')

model.load_state_dict(best_state)
model.eval()
preds = []
with torch.no_grad():
    for xb, _ in test_dl:
        preds.append(model(xb.to(device, non_blocking=True)).cpu().numpy())
y_mlp = np.concatenate(preds)

results['mlp'] = {
    'r2':   float(r2_score(y_test, y_mlp)),
    'mae':  float(mean_absolute_error(y_test, y_mlp)),
    'rmse': float(np.sqrt(mean_squared_error(y_test, y_mlp))),
    'training_time_seconds': time.time() - t0,
    'hidden_layers': '(100,)',
    'best_epoch': int(best_epoch),
    'best_val_mse': float(best_val),
    'n_train_used': int(len(X_train_s)),
    'hyperparams_source': 'scikit-learn MLPRegressor defaults (https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html); batch_size 4096 from published large-data tabular regime (pytorch_tabular default 1024; TabNet 4096 [Arik & Pfister 2021]; Goyal et al. 2017 verified up to 8192).',
    'hyperparams': {'hidden_layer_sizes': '(100,)', 'max_iter': 200, 'patience': 10, 'tol': 1e-4, 'lr': 1e-3, 'weight_decay': 1e-4, 'batch_size': 4096, 'optimizer': 'AdamW'},
}
print(f'\n  R^2  = {results["mlp"]["r2"]:.4f}')
print(f'  MAE  = {results["mlp"]["mae"]:.4f}')
print(f'  RMSE = {results["mlp"]["rmse"]:.4f}')
print(f'  Time = {results["mlp"]["training_time_seconds"]:.1f} s')
print(f'  Best epoch: {best_epoch}, best val MSE: {best_val:.4f}')

## Split conformal on top of Random Forest

Half the test set is used as a calibration partition for conformal, the other half as the evaluation partition. Same seed (42) as the GNN-side analyses.

In [ ]:
n_test = len(y_test)
perm = np.random.RandomState(42).permutation(n_test)
cal_idx, eval_idx = perm[:n_test // 2], perm[n_test // 2:]

residuals_cal = np.abs(y_test[cal_idx] - y_rf[cal_idx])
q_90 = float(np.quantile(residuals_cal, np.ceil((len(cal_idx) + 1) * 0.90) / len(cal_idx)))
q_95 = float(np.quantile(residuals_cal, np.ceil((len(cal_idx) + 1) * 0.95) / len(cal_idx)))

eval_residuals = np.abs(y_test[eval_idx] - y_rf[eval_idx])
picp_90 = float((eval_residuals <= q_90).mean() * 100)
picp_95 = float((eval_residuals <= q_95).mean() * 100)

results['random_forest_conformal'] = {
    'q_90': q_90,
    'q_95': q_95,
    'picp_90': picp_90,
    'picp_95': picp_95,
    'cal_size':  int(len(cal_idx)),
    'eval_size': int(len(eval_idx)),
}
print(f'  q_90 = {q_90:.3f}   PICP_90 = {picp_90:.2f}%')
print(f'  q_95 = {q_95:.3f}   PICP_95 = {picp_95:.2f}%')

## Save and print final results

**Copy the printed JSON block at the end of this cell and send it back for thesis integration.**

In [ ]:
results['scope'] = {
    'split_seed': 42,
    'train_test_split': '80/10/10 scenario-level (matches T8)',
    'n_train_graphs': len(train_set),
    'n_val_graphs':   len(val_set),
    'n_test_graphs':  len(test_set),
    'n_train_nodes':  int(len(X_train_s)),
    'n_test_nodes':   int(len(X_test_s)),
    'features':       FEATURE_NAMES,
}

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_PATH, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Saved results to: {OUT_PATH}\n')

print('=' * 70)
print('NON-GNN BASELINE RESULTS  (copy this whole block to share)')
print('=' * 70)
print(json.dumps(results, indent=2))
print('=' * 70)

print('\nCompact comparison vs T8 (R^2 = 0.5957, MAE = 3.957, RMSE = 7.118):')
print(f'  Random Forest      R^2 = {results["random_forest"]["r2"]:.4f}   MAE = {results["random_forest"]["mae"]:.3f}   RMSE = {results["random_forest"]["rmse"]:.3f}')
print(f'  Gradient Boosting  R^2 = {results["gradient_boosting"]["r2"]:.4f}   MAE = {results["gradient_boosting"]["mae"]:.3f}   RMSE = {results["gradient_boosting"]["rmse"]:.3f}')
print(f'  MLP                R^2 = {results["mlp"]["r2"]:.4f}   MAE = {results["mlp"]["mae"]:.3f}   RMSE = {results["mlp"]["rmse"]:.3f}')
print(f'\nRF + split conformal:  PICP_90 = {results["random_forest_conformal"]["picp_90"]:.2f}%   PICP_95 = {results["random_forest_conformal"]["picp_95"]:.2f}%')